In [1]:
!git clone https://github.com/Anand-786/llm-quantization-thesis.git
%cd /content/llm-quantization-thesis
!git clone https://github.com/mit-han-lab/smoothquant.git smoothquant_repo
!pip uninstall smoothquant -y
!cd smoothquant_repo && pip install -e .
!pip install -q transformers accelerate datasets zstandard tqdm
!pip install -q lm-eval==0.4.4
!pip install -q "datasets<3.0.0"

import sys
sys.path.insert(0, "/content/llm-quantization-thesis/smoothquant_repo")
sys.path.insert(0, "/content/llm-quantization-thesis")  # for experiments.task02_*

from google.colab import drive
drive.mount('/content/drive')

# Copy Task 01 max scales + Task 02 percentile scales locally
!mkdir -p smoothquant_repo/act_scales
!cp /content/drive/MyDrive/thesis_results/act_scales/opt-1.3b.pt smoothquant_repo/act_scales/

!mkdir -p act_percentiles/opt-1.3b
!cp /content/drive/MyDrive/thesis_results/act_percentiles/opt-1.3b/*.pt act_percentiles/opt-1.3b/

!nvidia-smi
!ls -la smoothquant_repo/act_scales/ act_percentiles/opt-1.3b/
!python -c "from smoothquant.smooth import smooth_lm; print('smoothquant OK')"
!python -c "from experiments.task02_percentile_smoothing.percentile_smooth import smooth_lm_pct; print('percentile smooth OK')"
!python -c "import importlib.metadata; print('lm_eval', importlib.metadata.version('lm_eval'))"

Cloning into 'llm-quantization-thesis'...
remote: Enumerating objects: 153, done.
remote: Counting objects: 100% (153/153), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 153 (delta 58), reused 137 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (153/153), 4.86 MiB | 22.52 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/content/llm-quantization-thesis
Cloning into 'smoothquant_repo'...
remote: Enumerating objects: 352, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 352 (delta 120), reused 90 (delta 90), pack-reused 183 (from 1)
Receiving objects: 100% (352/352), 6.80 MiB | 4.28 MiB/s, done.
Resolving deltas: 100% (202/202), done.
Obtaining file:///content/llm-quantization-thesis/smoothquant_repo
  Preparing metadata (setup.py) ... done
  Running setup.py develop for smoothquant
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (

In [7]:
# Paths
MODEL = "facebook/opt-1.3b"
SCRIPT = "experiments/task03_zero_shot_eval/opt_1_3b/run_zero_shot_t3.py"
MAX_SCALES = "smoothquant_repo/act_scales/opt-1.3b.pt"

# Alphas (locked from Task 01 PPL sweep on 1.3B)
ALPHA_O1 = 0.5
ALPHA_O2 = 0.5
ALPHA_C_MAX = 0.9

# Percentile-smoothing knobs — REPLACE with Task 02's PPL winner before running
P_PCT     = 0.999
ALPHA_PCT = 0.9
PCT_SCALES = f"act_percentiles/opt-1.3b/p{P_PCT}.pt"

BATCH = 8  # drop to 4 or 2 if HellaSwag OOMs

OUT_DIR = "results/task03"
!mkdir -p {OUT_DIR}
print(f"Percentile config -> p={P_PCT}, alpha={ALPHA_PCT}, file={PCT_SCALES}")

Percentile config -> p=0.999, alpha=0.9, file=act_percentiles/opt-1.3b/p0.999.pt


In [3]:
!python {SCRIPT} \
    --model_path {MODEL} \
    --config_label FP16 \
    --batch_size {BATCH} \
    --save_json {OUT_DIR}/opt-1.3b_zeroshot_FP16.json

  Config:        FP16
  Model:         facebook/opt-1.3b
  Smooth:        False (max, alpha=0.5, p_w=1.0)
  Quant:         False (W=per_channel, A=per_token, bmm=True)
  Tasks:         ['lambada_openai', 'hellaswag', 'piqa', 'winogrande', 'openbookqa', 'rte', 'copa']
  Batch size:    8
2026-05-08:00:13:49,059 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/facebook/opt-1.3b/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08:00:13:49,065 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/opt-1.3b/3f5c25d0bc631cb57ac65913f76e22c2dfb61d62/config.json "HTTP/1.1 200 OK"
2026-05-08:00:13:49,071 INFO     [_client.py:1025] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/facebook/opt-1.3b/3f5c25d0bc631cb57ac65913f76e22c2dfb61d62/config.json "HTTP/1.1 200 OK"
config.json: 100% 653/653 [00:00<00:00, 3.45MB/s]
2026-05-08:00:13:49,317 INFO     [_client.py:1025] HTTP Request: HEAD https://hu

In [4]:
!python {SCRIPT} \
    --model_path {MODEL} \
    --quantize \
    --weight_quant per_tensor --act_quant per_tensor \
    --config_label W8A8-naive \
    --batch_size {BATCH} \
    --save_json {OUT_DIR}/opt-1.3b_zeroshot_W8A8-naive.json

  Config:        W8A8-naive
  Model:         facebook/opt-1.3b
  Smooth:        False (max, alpha=0.5, p_w=1.0)
  Quant:         True (W=per_tensor, A=per_tensor, bmm=True)
  Tasks:         ['lambada_openai', 'hellaswag', 'piqa', 'winogrande', 'openbookqa', 'rte', 'copa']
  Batch size:    8
2026-05-08:00:23:21,152 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/facebook/opt-1.3b/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08:00:23:21,158 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/opt-1.3b/3f5c25d0bc631cb57ac65913f76e22c2dfb61d62/config.json "HTTP/1.1 200 OK"
2026-05-08:00:23:21,394 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/facebook/opt-1.3b/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08:00:23:21,399 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/opt-1.3b/3f5c25d0bc631cb

In [5]:
!python {SCRIPT} \
    --model_path {MODEL} \
    --smooth --smooth_method max --alpha {ALPHA_O1} \
    --act_scales_path {MAX_SCALES} \
    --quantize \
    --weight_quant per_tensor --act_quant per_token \
    --config_label SQ-O1-max \
    --batch_size {BATCH} \
    --save_json {OUT_DIR}/opt-1.3b_zeroshot_SQ-O1-max.json

  Config:        SQ-O1-max
  Model:         facebook/opt-1.3b
  Smooth:        True (max, alpha=0.5, p_w=1.0)
  Quant:         True (W=per_tensor, A=per_token, bmm=True)
  Tasks:         ['lambada_openai', 'hellaswag', 'piqa', 'winogrande', 'openbookqa', 'rte', 'copa']
  Batch size:    8
2026-05-08:00:34:09,729 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/facebook/opt-1.3b/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08:00:34:09,735 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/opt-1.3b/3f5c25d0bc631cb57ac65913f76e22c2dfb61d62/config.json "HTTP/1.1 200 OK"
2026-05-08:00:34:09,969 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/facebook/opt-1.3b/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08:00:34:09,969 WARNING  [_http.py:904] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher ra

In [6]:
!python {SCRIPT} \
    --model_path {MODEL} \
    --smooth --smooth_method max --alpha {ALPHA_O2} \
    --act_scales_path {MAX_SCALES} \
    --quantize \
    --weight_quant per_tensor --act_quant per_tensor \
    --config_label SQ-O2-max \
    --batch_size {BATCH} \
    --save_json {OUT_DIR}/opt-1.3b_zeroshot_SQ-O2-max.json

  Config:        SQ-O2-max
  Model:         facebook/opt-1.3b
  Smooth:        True (max, alpha=0.5, p_w=1.0)
  Quant:         True (W=per_tensor, A=per_tensor, bmm=True)
  Tasks:         ['lambada_openai', 'hellaswag', 'piqa', 'winogrande', 'openbookqa', 'rte', 'copa']
  Batch size:    8
2026-05-08:00:44:09,264 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/facebook/opt-1.3b/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08:00:44:09,265 WARNING  [_http.py:904] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-08:00:44:09,270 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/opt-1.3b/3f5c25d0bc631cb57ac65913f76e22c2dfb61d62/config.json "HTTP/1.1 200 OK"
2026-05-08:00:44:09,498 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/facebook/opt-1.3b/resolve/main/tokenizer_config.json "

In [8]:
!python {SCRIPT} \
    --model_path {MODEL} \
    --smooth --smooth_method percentile --alpha {ALPHA_PCT} --p_w {P_PCT} \
    --act_scales_path {PCT_SCALES} \
    --quantize \
    --weight_quant per_channel --act_quant per_token \
    --config_label SQ-C-pct \
    --batch_size {BATCH} \
    --save_json {OUT_DIR}/opt-1.3b_zeroshot_SQ-C-pct.json

  Config:        SQ-C-pct
  Model:         facebook/opt-1.3b
  Smooth:        True (percentile, alpha=0.9, p_w=0.999)
  Quant:         True (W=per_channel, A=per_token, bmm=True)
  Tasks:         ['lambada_openai', 'hellaswag', 'piqa', 'winogrande', 'openbookqa', 'rte', 'copa']
  Batch size:    8
2026-05-08:00:54:03,663 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/facebook/opt-1.3b/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08:00:54:03,669 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/opt-1.3b/3f5c25d0bc631cb57ac65913f76e22c2dfb61d62/config.json "HTTP/1.1 200 OK"
2026-05-08:00:54:03,909 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/facebook/opt-1.3b/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08:00:54:03,915 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/opt-1.3b/3f5c25d0b

In [9]:
!mkdir -p /content/drive/MyDrive/thesis_results/task03
!cp {OUT_DIR}/opt-1.3b_zeroshot_*.json /content/drive/MyDrive/thesis_results/task03/

import json, glob

TASKS = ["lambada_openai", "hellaswag", "piqa", "winogrande", "openbookqa", "rte", "copa"]
PRIMARY = {
    "lambada_openai": "acc,none",
    "hellaswag":      "acc_norm,none",
    "piqa":           "acc_norm,none",
    "winogrande":     "acc,none",
    "openbookqa":     "acc_norm,none",
    "rte":            "acc,none",
    "copa":           "acc,none",
}

# Order matches reading flow: anchors first, then paper, then ours
ORDER = ["FP16", "W8A8-naive", "SQ-O1-max", "SQ-O2-max", "SQ-C-pct"]

rows_by_label = {}
for f in sorted(glob.glob(f"{OUT_DIR}/opt-1.3b_zeroshot_*.json")):
    r = json.load(open(f))
    label = r["config_label"]
    row = {"config": label}
    for t in TASKS:
        m = r["results"].get(t, {})
        v = m.get(PRIMARY[t])
        if v is None:
            for k, val in m.items():
                if isinstance(val, (int, float)):
                    v = val
                    break
        row[t] = v
    nums = [row[t] for t in TASKS if isinstance(row[t], (int, float))]
    row["avg"] = sum(nums) / len(nums) if nums else None
    rows_by_label[label] = row

rows = [rows_by_label[l] for l in ORDER if l in rows_by_label]

header = ["config"] + TASKS + ["avg"]
print("  ".join(f"{h:>14}" for h in header))
print("-" * (16 * len(header)))
for row in rows:
    cells = [f"{row['config']:>14}"]
    for t in TASKS + ["avg"]:
        v = row.get(t)
        cells.append(f"{v:>14.4f}" if isinstance(v, (int, float)) else f"{'-':>14}")
    print("  ".join(cells))

        config  lambada_openai       hellaswag            piqa      winogrande      openbookqa             rte            copa             avg
------------------------------------------------------------------------------------------------------------------------------------------------
          FP16          0.5787          0.5370          0.7236          0.5943          0.3320          0.5235          0.8000          0.5842
    W8A8-naive          0.5432          0.5174          0.7078          0.5714          0.3160          0.5668          0.7900          0.5732
     SQ-O1-max          0.5754          0.5342          0.7231          0.5880          0.3380          0.5199          0.8000          0.5826
     SQ-O2-max          0.5570          0.5310          0.7089          0.5825          0.3300          0.5343          0.8100          0.5791
      SQ-C-pct          0.5744          0.5365          0.7242          0.5927          0.3280          0.5271          0.8100          0.58